In [ ]:
!cp -r /content/drive/MyDrive/FightAttention/Stage2 /content/Stage2

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

"""
layer_10 torch.Size([1, 512, 7, 10])
layer_16 torch.Size([1, 128, 28, 40])
layer_19 torch.Size([1, 256, 14, 20])
layer_22 torch.Size([1, 512, 7, 10])

gap_feat = torch.mean(feat, dim=[2,3])  # Global Average Pooling

sequence.append(gap_feat)

if len(sequence) == T:
    input_seq = torch.stack(sequence, dim=1)  # (B, T, C)
    output = model(input_seq)
"""

class GAPGRU(torch.nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
        super().__init__()
        self.gru = torch.nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = torch.nn.Dropout(dropout)
        self.fc = torch.nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, h_n = self.gru(x)
        out = self.dropout(h_n[-1])
        return self.fc(out)
    
class GAPConv1D(torch.nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, dropout=0.3, pool_output_size=4):
        super().__init__()
        self.conv = torch.nn.Sequential(
            torch.nn.Conv1d(input_size, hidden_size, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Conv1d(hidden_size, hidden_size, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
        )
        self.pool = torch.nn.AdaptiveAvgPool1d(pool_output_size)  # AAP1d: compress to fixed output size
        self.fc = torch.nn.Linear(hidden_size * pool_output_size, num_classes)  # flattened size

    def forward(self, x):
        # x: (batch_size, seq_length, input_size)
        x = x.permute(0, 2, 1)          # -> (batch, input_size, seq_length)
        x = self.conv(x)                 # -> (batch, hidden_size, seq_length)
        x = self.pool(x)                 # -> (batch, hidden_size, pool_output_size)  ← AAP1d
        x = x.flatten(start_dim=1)       # -> (batch, hidden_size * pool_output_size)
        return self.fc(x)
    
class AAP1DConv1D(torch.nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, dropout=0.3, pool_output_size=4):
        super().__init__()
        self.conv = torch.nn.Sequential(
            torch.nn.Conv1d(input_size, hidden_size, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Conv1d(hidden_size, hidden_size, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
        )
        self.pool = torch.nn.AdaptiveAvgPool1d(pool_output_size)  # AAP1d: compress to fixed output size
        self.fc = torch.nn.Linear(hidden_size * pool_output_size, num_classes)  # flattened size

    def forward(self, x):
        # x: (batch_size, seq_length, input_size)
        x = x.permute(0, 2, 1)          # -> (batch, input_size, seq_length)
        x = self.conv(x)                 # -> (batch, hidden_size, seq_length)
        x = self.pool(x)                 # -> (batch, hidden_size, pool_output_size)  ← AAP1d
        x = x.flatten(start_dim=1)       # -> (batch, hidden_size * pool_output_size)
        return self.fc(x)

In [ ]:
class FeatureDataset(Dataset):
    def __init__(self, npz_path):
        data = np.load(npz_path)
        self.features = torch.tensor(data['features'], dtype=torch.float32)  # [N, seq_len, feat_dim]
        self.labels   = torch.tensor(data['labels'],   dtype=torch.long)     # [N]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]  # [seq_len, feat_dim], scalar


train_dataset = FeatureDataset("/content/Stage2/train_features_16_16.npz")
val_dataset   = FeatureDataset("/content/Stage2/val_features_16_16_.npz")

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)

# Sanity check
features, labels = next(iter(train_loader))
print(features.shape)  # [16, 8, 512]
print(labels.shape)    # [16]

In [ ]:
# Hyperparameters
input_size  = features.shape[-1]
hidden_size = 256
num_layers  = 2
num_classes = 1          # binary → single logit
num_epochs  = 1
lr          = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model= GAPConv1D(input_size, hidden_size, num_layers, num_classes).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

for epoch in range(num_epochs):
    # --- Train ---
    model.train()
    train_loss, train_correct = 0, 0
    for feats, labels in train_loader:
        feats, labels = feats.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model_gru(feats).squeeze(1)          # [B, 1] → [B]
        loss = criterion(outputs, labels.float())
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        train_correct += ((outputs > 0).long() == labels).sum().item()

    # --- Val ---
    model_gru.eval()
    val_loss, val_correct = 0, 0
    with torch.no_grad():
        for feats, labels in val_loader:
            feats, labels = feats.to(device), labels.to(device)
            outputs = model_gru(feats).squeeze(1)      # [B, 1] → [B]
            val_loss    += criterion(outputs, labels.float()).item()
            val_correct += ((outputs > 0).long() == labels).sum().item()

    print(f"Epoch [{epoch+1:02d}/{num_epochs}] "
          f"Train Loss: {train_loss/len(train_loader):.4f}  Acc: {train_correct/len(train_dataset):.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f}  Acc: {val_correct/len(val_dataset):.4f}")

# Save model
torch.save(model_gru.state_dict(), "/content/Stage2/gru_model.pt")
print("Model saved.")

In [ ]:
#!cp /content/Stage2/gru_model.pt /content/drive/MyDrive/FightAttention/Stage2/gru_model.pt